# BoomBikes Shared Bike Demand Prediction
## Multiple Linear Regression — End-to-End Analysis

**Business Goal:** Model daily bike-sharing demand using available independent variables so that management can understand demand dynamics, manipulate business strategy, and prepare for post-pandemic recovery.

---
**Table of Contents**
1. [Problem Statement](#1-problem-statement)
2. [Import Libraries](#2-import-libraries)
3. [Load & Understand the Data](#3-load--understand-the-data)
4. [Data Preparation](#4-data-preparation)
5. [Exploratory Data Analysis (EDA)](#5-exploratory-data-analysis)
6. [Dummy Encoding & Train/Test Split](#6-dummy-encoding--traintest-split)
7. [Feature Scaling](#7-feature-scaling)
8. [Feature Selection — RFE](#8-feature-selection--rfe)
9. [Model Building — Iterative OLS](#9-model-building--iterative-ols)
10. [Model Diagnostics](#10-model-diagnostics)
11. [Prediction & Evaluation on Test Set](#11-prediction--evaluation-on-test-set)
12. [Conclusions & Business Insights](#12-conclusions--business-insights)


---
## 1. Problem Statement

BoomBikes, a US bike-sharing provider, suffered significant revenue losses during the COVID-19 pandemic. To plan their post-lockdown strategy, they want to:

- **Identify** which variables significantly predict daily shared-bike demand.
- **Quantify** how well those variables describe demand variation.

**Dataset:** Daily bike demand data (730 rows) across the American market for 2018–2019, with meteorological and calendar features.

**Target Variable:** `cnt` — total daily rentals (casual + registered users)


---
## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.facecolor'] = '#F8F9FA'
plt.rcParams['axes.facecolor']   = '#FFFFFF'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.3

print("All libraries imported successfully.")


---
## 3. Load & Understand the Data

In [ ]:
df = pd.read_csv('day.csv')
print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")


---
## 4. Data Preparation

### 4.1 Drop Irrelevant / Leakage Columns

| Column | Reason for dropping |
|--------|---------------------|
| `instant` | Row index — no predictive value |
| `dteday` | Date string — captured by `yr`, `mnth`, `weekday` |
| `casual` | Sub-component of `cnt` → data leakage |
| `registered` | Sub-component of `cnt` → data leakage |


In [ ]:
df.drop(['instant', 'dteday', 'casual', 'registered'], axis=1, inplace=True)
print(f"Shape after dropping columns: {df.shape}")
df.columns.tolist()


### 4.2 Map Numeric Codes to Categorical Labels

`season` and `weathersit` store numeric codes (1–4) that imply a **false ordinal** relationship. They must be converted to string labels before dummy encoding.


In [ ]:
# Data Dictionary mappings
season_map     = {1: 'Spring', 2: 'Summer', 3: 'Fall',  4: 'Winter'}
weathersit_map = {1: 'Clear',  2: 'Mist',   3: 'Light_Snow_Rain', 4: 'Heavy_Rain'}
month_map      = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr',  5:'May',  6:'Jun',
                  7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
weekday_map    = {0:'Sun', 1:'Mon', 2:'Tue', 3:'Wed', 4:'Thu', 5:'Fri', 6:'Sat'}

df['season']     = df['season'].map(season_map)
df['weathersit'] = df['weathersit'].map(weathersit_map)
df['mnth']       = df['mnth'].map(month_map)
df['weekday']    = df['weekday'].map(weekday_map)

# yr: keep as 0/1 — captures year-on-year growth trend
# holiday & workingday: already binary
df['holiday']    = df['holiday'].astype(int)
df['workingday'] = df['workingday'].astype(int)

print("Unique values per categorical column:")
for col in ['season', 'weathersit', 'mnth', 'weekday']:
    print(f"  {col}: {sorted(df[col].unique())}")


In [ ]:
df.head(3)


---
## 5. Exploratory Data Analysis

### 5.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['cnt'], bins=30, color='#2563EB', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Daily Bike Demand (cnt)', fontweight='bold')
axes[0].set_xlabel('Daily Rides'); axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['cnt'], vert=False,
                boxprops=dict(color='#2563EB'),
                medianprops=dict(color='#EF4444', linewidth=2),
                whiskerprops=dict(color='#2563EB'),
                capprops=dict(color='#2563EB'))
axes[1].set_title('Boxplot of Daily Bike Demand', fontweight='bold')
axes[1].set_xlabel('Daily Rides')

plt.suptitle('Target Variable: cnt', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean  : {df['cnt'].mean():.0f}")
print(f"Median: {df['cnt'].median():.0f}")
print(f"Std   : {df['cnt'].std():.0f}")
print(f"Min   : {df['cnt'].min()}")
print(f"Max   : {df['cnt'].max()}")


### 5.2 Year-on-Year Monthly Demand Trend

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 5))
for yr_val, color, lbl in [(0, '#2563EB', '2018'), (1, '#F59E0B', '2019')]:
    d = df[df['yr'] == yr_val].groupby('mnth')['cnt'].mean().reindex(month_order)
    ax.plot(range(12), d.values, marker='o', color=color, linewidth=2.5, label=lbl, markersize=7)

ax.fill_between(range(12),
    df[df['yr']==0].groupby('mnth')['cnt'].mean().reindex(month_order).values,
    df[df['yr']==1].groupby('mnth')['cnt'].mean().reindex(month_order).values,
    alpha=0.1, color='#10B981', label='YoY Growth')

ax.set_xticks(range(12)); ax.set_xticklabels(month_order)
ax.set_title('Monthly Average Bike Demand: 2018 vs 2019', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Daily Rides'); ax.legend(fontsize=11)
plt.tight_layout(); plt.show()


### 5.3 Demand by Categorical Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
COLORS = ['#2563EB', '#10B981', '#F59E0B', '#EF4444']

# Season
season_avg = df.groupby('season')['cnt'].mean().reindex(['Spring','Summer','Fall','Winter'])
bars = axes[0].bar(season_avg.index, season_avg.values, color=COLORS, edgecolor='white', width=0.6)
for bar in bars:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Avg Demand by Season', fontweight='bold')
axes[0].set_ylabel('Avg Daily Rides')

# Weather
weather_avg = df.groupby('weathersit')['cnt'].mean().reindex(['Clear','Mist','Light_Snow_Rain'])
bars = axes[1].bar(weather_avg.index, weather_avg.values,
                   color=['#10B981','#2563EB','#F59E0B'], edgecolor='white', width=0.5)
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Avg Demand by Weather Situation', fontweight='bold')

# Working Day
wd_avg = df.groupby('workingday')['cnt'].mean()
bars = axes[2].bar(['Non-Working Day','Working Day'], wd_avg.values,
                   color=['#8B5CF6','#2563EB'], edgecolor='white', width=0.5)
for bar in bars:
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=9, fontweight='bold')
axes[2].set_title('Avg Demand: Working vs Non-Working Day', fontweight='bold')

plt.suptitle('Demand by Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 5.4 Demand vs Continuous Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cont_vars  = ['temp', 'hum', 'windspeed']
colors     = ['#2563EB', '#10B981', '#F59E0B']
xlabels    = ['Temperature (°C)', 'Humidity (%)', 'Wind Speed (km/h)']

for i, (var, color, xlabel) in enumerate(zip(cont_vars, colors, xlabels)):
    axes[i].scatter(df[var], df['cnt'], alpha=0.4, color=color, s=15)
    z = np.polyfit(df[var], df['cnt'], 1)
    x_line = np.linspace(df[var].min(), df[var].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), color='#EF4444', linewidth=2)
    axes[i].set_title(f'{xlabel} vs Demand', fontweight='bold')
    axes[i].set_xlabel(xlabel); axes[i].set_ylabel('Daily Rides')

plt.suptitle('Continuous Features vs Demand (cnt)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 5.5 Correlation Heatmap (Numeric Features)

In [ ]:
num_cols = ['temp', 'atemp', 'hum', 'windspeed', 'yr', 'holiday', 'workingday', 'cnt']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 11}, ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("Key observations:")
print(f"  temp  ↔ cnt  : {corr.loc['temp','cnt']:.2f}  (strong positive)")
print(f"  atemp ↔ cnt  : {corr.loc['atemp','cnt']:.2f}  (strong positive)")
print(f"  hum   ↔ cnt  : {corr.loc['hum','cnt']:.2f}  (moderate negative)")
print(f"  temp  ↔ atemp: {corr.loc['temp','atemp']:.2f}  (⚠ near-perfect — multicollinearity risk)")


---
## 6. Dummy Encoding & Train/Test Split

`drop_first=True` is used to avoid the **dummy variable trap** (perfect multicollinearity among dummies).


In [ ]:
cat_cols = ['season', 'mnth', 'weekday', 'weathersit']
df_model = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df_model = df_model.astype(float)

print(f"Shape after encoding: {df_model.shape}")
print(f"Columns ({len(df_model.columns)}): {list(df_model.columns)}")


In [ ]:
y = df_model['cnt']
X = df_model.drop('cnt', axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train size : {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  size : {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.0f}%)")


---
## 7. Feature Scaling

Min-Max scaling is applied to continuous features. The scaler is **fit only on the training set** to prevent data leakage into the test set.


In [ ]:
num_cols = ['temp', 'atemp', 'hum', 'windspeed']
scaler = MinMaxScaler()

X_train = X_train.copy(); X_test = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print("Scaling complete. Sample of scaled training data:")
X_train[num_cols].describe().round(3)


---
## 8. Feature Selection — Recursive Feature Elimination (RFE)

RFE uses sklearn's `LinearRegression` to rank features by importance and select the top 15.


In [ ]:
lm_sk = LinearRegression()
rfe   = RFE(lm_sk, n_features_to_select=15)
rfe.fit(X_train, y_train)

rfe_support = pd.DataFrame({
    'Feature'  : X_train.columns,
    'Selected' : rfe.support_,
    'Ranking'  : rfe.ranking_
}).sort_values('Ranking')

print("RFE Results:")
print(rfe_support.to_string(index=False))


In [ ]:
rfe_features = X_train.columns[rfe.support_].tolist()
print(f"\nSelected features ({len(rfe_features)}):")
print(rfe_features)


---
## 9. Model Building — Iterative OLS

### Strategy
1. Start with RFE-selected features.
2. Drop `atemp` immediately — it has a **Pearson r ≈ 0.99** with `temp`, causing severe multicollinearity.
3. Iteratively remove features with **p-value > 0.05** or **VIF > 5**, dropping the worst offender each round.
4. Stop when all remaining features are statistically significant and VIF < 5.


In [ ]:
def build_ols(X, y):
    """Fit OLS model with constant."""
    Xc = sm.add_constant(X.astype(float))
    return sm.OLS(y.astype(float), Xc).fit()

def calc_vif(X):
    """Compute VIF for all features."""
    Xc = sm.add_constant(X.astype(float))
    return pd.DataFrame({
        'Feature': Xc.columns,
        'VIF'    : [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]
    })


In [ ]:
cols = rfe_features.copy()

# Step 1: Remove atemp (multicollinear with temp)
if 'atemp' in cols:
    cols.remove('atemp')
    print("Dropped 'atemp' — near-perfect collinearity with 'temp' (r ≈ 0.99)")

# Step 2: Iterative elimination
print("\nIterative elimination log:")
for iteration in range(15):
    model    = build_ols(X_train[cols], y_train)
    vif      = calc_vif(X_train[cols])

    high_vif   = [f for f in vif[vif['VIF'] > 5]['Feature'].tolist() if f != 'const']
    pvals_nc   = model.pvalues.drop('const', errors='ignore')
    high_p     = pvals_nc[pvals_nc > 0.05].index.tolist()
    candidates = list(set(high_p + high_vif))

    if not candidates:
        print(f"  Iter {iteration}: ✓ All features significant & VIF < 5. Stopping.")
        break

    to_drop = pvals_nc[pvals_nc.index.isin(candidates)].idxmax()
    print(f"  Iter {iteration}: Dropped '{to_drop}'  (p = {pvals_nc[to_drop]:.4f})")
    cols.remove(to_drop)

print(f"\nFinal feature set ({len(cols)} features):")
print(cols)


In [ ]:
# Final model summary
print(model.summary())


In [ ]:
# VIF check on final model
vif_final = calc_vif(X_train[cols])
vif_final = vif_final[vif_final['Feature'] != 'const'].reset_index(drop=True)
print("VIF — Final Model (all should be < 5):")
print(vif_final.to_string(index=False))


---
## 10. Model Diagnostics

### OLS Assumptions to Verify
1. **Linearity** — residuals randomly scattered around 0 vs fitted values
2. **Normality of residuals** — Q-Q plot hugs the diagonal; histogram is bell-shaped
3. **Homoscedasticity** — no funnel pattern in residuals vs fitted
4. **No autocorrelation** — Durbin-Watson statistic ≈ 2


In [ ]:
residuals = y_train - model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Residuals vs Fitted
axes[0].scatter(model.fittedvalues, residuals, alpha=0.4, color='#2563EB', s=20)
axes[0].axhline(0, color='#EF4444', linewidth=1.5, linestyle='--')
axes[0].set_title('Residuals vs Fitted Values', fontweight='bold')
axes[0].set_xlabel('Fitted Values'); axes[0].set_ylabel('Residuals')

# Q-Q Plot
(osm, osr), (slope, intercept, _) = stats.probplot(residuals, dist="norm")
axes[1].scatter(osm, osr, alpha=0.5, color='#10B981', s=20)
axes[1].plot(osm, slope * np.array(osm) + intercept, color='#EF4444', linewidth=1.5)
axes[1].set_title('Normal Q-Q Plot of Residuals', fontweight='bold')
axes[1].set_xlabel('Theoretical Quantiles'); axes[1].set_ylabel('Sample Quantiles')

# Histogram of residuals
axes[2].hist(residuals, bins=30, color='#F59E0B', edgecolor='white', alpha=0.85)
axes[2].set_title('Distribution of Residuals', fontweight='bold')
axes[2].set_xlabel('Residuals'); axes[2].set_ylabel('Frequency')

plt.suptitle('Model Diagnostics — Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Durbin-Watson statistic: {sm.stats.stattools.durbin_watson(residuals):.3f}")
print("(Value close to 2.0 → no significant autocorrelation)")


---
## 11. Prediction & Evaluation on Test Set


In [ ]:
X_test_c = sm.add_constant(X_test[cols].astype(float))
y_pred   = model.predict(X_test_c)

r2_train  = model.rsquared
adj_r2    = model.rsquared_adj
r2_test   = r2_score(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 45)
print("       MODEL PERFORMANCE METRICS")
print("=" * 45)
print(f"  Train R²         : {r2_train:.4f}")
print(f"  Train Adj. R²    : {adj_r2:.4f}")
print(f"  Test  R²         : {r2_test:.4f}")
print(f"  Test  RMSE       : {rmse_test:.2f} rides/day")
print(f"  Train-Test R² gap: {abs(r2_train - r2_test):.4f}  (< 0.05 = no overfit)")
print("=" * 45)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(y_test, y_pred, alpha=0.5, color='#2563EB', s=30, label='Test Predictions')
lims = [min(y_test.min(), y_pred.min()) - 300,
        max(y_test.max(), y_pred.max()) + 300]
ax.plot(lims, lims, color='#EF4444', linestyle='--', linewidth=1.8, label='Perfect Fit Line')
ax.set_xlabel('Actual Daily Rides', fontsize=12)
ax.set_ylabel('Predicted Daily Rides', fontsize=12)
ax.set_title(f'Actual vs Predicted — Test Set  |  R² = {r2_test:.3f}', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Coefficient impact chart
coef_df = (pd.DataFrame({'Feature': model.params.index, 'Coefficient': model.params.values})
             .query("Feature != 'const'")
             .sort_values('Coefficient'))

fig, ax = plt.subplots(figsize=(11, 7))
colors = ['#EF4444' if c < 0 else '#2563EB' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8)

for bar in bars:
    w = bar.get_width()
    ax.text(w + (40 if w >= 0 else -40), bar.get_y() + bar.get_height()/2,
            f'{w:.0f}', va='center', ha='left' if w >= 0 else 'right', fontsize=8.5)

ax.set_title('Model Coefficients — Impact on Daily Bike Demand', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#2563EB', label='Positive effect'),
    plt.Rectangle((0,0),1,1, color='#EF4444', label='Negative effect')
], loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()


---
## 12. Conclusions & Business Insights

### 12.1 Model Equation

```
cnt = 3235
      + 1973 × yr
      + 3717 × temp_scaled
      − 1357 × hum_scaled
      − 1199 × windspeed_scaled
      − 1265 × season_Spring
      +  742 × season_Winter
      +  458 × mnth_Mar
      +  434 × mnth_Sep
      −  507 × mnth_Dec
      −  526 × mnth_Jul
      −  693 × mnth_Nov
      +  202 × weekday_Sat
      − 2055 × weathersit_Light_Snow_Rain
      −  433 × weathersit_Mist
```
*(Continuous features are Min-Max scaled; Fall & Clear weather are reference categories)*

---

### 12.2 Significant Variables

| Rank | Feature | Coefficient | Effect |
|------|---------|-------------|--------|
| 1 | `temp` (temperature) | +3,717 | Strongest positive driver |
| 2 | `yr` (year) | +1,973 | Year-on-year platform growth |
| 3 | `weathersit_Light_Snow_Rain` | −2,055 | Strongest negative driver |
| 4 | `season_Spring` | −1,265 | Demand trough vs Fall |
| 5 | `hum` (humidity) | −1,357 | Moderate suppressor |
| 6 | `windspeed` | −1,199 | Moderate suppressor |
| 7 | `mnth_Nov` | −693 | November underperforms |
| 8 | `season_Winter` | +742 | Winter above Spring |

---

### 12.3 Business Recommendations

**Conclusion 1 — Year-on-year growth is real and should be planned for.**  
The `yr` coefficient of +1,973 confirms the platform is growing. BoomBikes should invest in fleet expansion before each new season rather than treating demand as static.

**Conclusion 2 — Temperature is the single biggest positive lever.**  
With a coefficient of +3,717, warm days are significantly more profitable. BoomBikes should maximise fleet availability heading into late summer and Fall (the peak season) and consider dynamic pricing to capture warm-day demand.

**Conclusion 3 — Bad weather is the biggest demand killer — but it's forecastable.**  
Light Snow/Rain events cut demand by ~2,055 rides/day (the largest single effect in the model). High humidity and wind speed also suppress demand. BoomBikes can integrate weather APIs into operations to pre-position bikes and launch rainy-day promotions.

**Conclusion 4 — Spring is a significant trough; targeted campaigns can close the gap.**  
Spring demand is ~1,265 rides/day below Fall. November and December also underperform. Discounted seasonal passes and marketing campaigns in these months can smooth out revenue troughs and improve annual stability.

---

### 12.4 Model Performance Summary

| Metric | Value |
|--------|-------|
| Train R² | **0.833** |
| Train Adjusted R² | **0.829** |
| Test R² | **0.851** |
| Test RMSE | **~714 rides/day** |
| No. of features | **14** |
| Durbin-Watson | **≈ 2.01** |

The model explains **83–85% of variance** in daily bike demand with no signs of overfitting (test R² ≥ train R²), all features statistically significant (p < 0.05), and all VIFs below 5.
